In [2]:
import pandas as pd
import numpy as np

#生成日期范围
dates = pd.date_range('2024-01-01',periods = 100,freq = 'D')
#创建data frame
df = pd.DataFrame({
    'date':dates,
    'price': 100 + np.cumsum(np.random.randn(100)),
    'volume': np.random.randint(1e6, 1e7,100)
})
#设置索引
df = df.set_index('date')
#提取时间信息：年月日
df['year']=df.index.year
df['month']=df.index.month
df['weekday']=df.index.weekday
#筛选一月数据
jan_data = df[df.index.month == 1]
#Resampling
df.resample('W').agg(['mean','sum'])
print(df.resample('ME').last()['price']/df.resample('ME').first()['price']-1)
df.resample('QE').last()

date
2024-01-31   -0.007346
2024-02-29    0.030319
2024-03-31    0.018411
2024-04-30    0.033422
Freq: ME, Name: price, dtype: float64


,price,volume,year,month,weekday
date,,,,,
2024-03-31,104.631470,7874477,2024,3,6
2024-06-30,107.593728,5628712,2024,4,1


In [15]:
# 生成 252 个交易日（约一年）
dates = pd.date_range('2024-01-01', periods=252, freq='B')
df = pd.DataFrame({
    'price': 100 + np.cumsum(np.random.randn(252) * 2)  # 模拟股价
}, index = dates)
df['return'] = df['price']/df['price'].shift(1) - 1 
df['prev_return'] = df['return'].shift(1)
df['post_return'] = df['return'].shift(-1)
df['MA20'] = df['price'].rolling(window = 20).mean()
df['vol_20d'] = df['return'].rolling(window = 20).std() * np.sqrt(252)
df['ret_5d'] = df['return'].rolling(window = 5).sum()

In [32]:
market = pd.DataFrame({'market_return': np.random.randn(252)}, index=dates)
df['bull'] = df['price'] > df['MA20']
df['market_return'] = market['market_return']
#rolling 60日beta
def calc_beta(returns, market):
    cov = np.cov(returns, market)[0,1]
    var = np.var(market)
    return cov / var
df['beta'] = df['return'].rolling(60).apply(
    lambda x: calc_beta(x,  df.loc[x.index, 'market_return'])
)
print(df)

                 price    return  prev_return  post_return        MA20  \
2024-01-01  100.208477       NaN          NaN    -0.007258         NaN   
2024-01-02   99.481136 -0.007258          NaN     0.022613         NaN   
2024-01-03  101.730689  0.022613    -0.007258     0.000495         NaN   
2024-01-04  101.781034  0.000495     0.022613    -0.011836         NaN   
2024-01-05  100.576397 -0.011836     0.000495     0.025252         NaN   
...                ...       ...          ...          ...         ...   
2024-12-11  121.324157  0.007190     0.024906    -0.014169  117.908738   
2024-12-12  119.605081 -0.014169     0.007190     0.014177  118.246327   
2024-12-13  121.300732  0.014177    -0.014169    -0.036882  118.728057   
2024-12-16  116.826951 -0.036882     0.014177    -0.004645  118.865636   
2024-12-17  116.284315 -0.004645    -0.036882          NaN  118.918763   

             vol_20d    ret_5d   bull  market_return      beta  
2024-01-01       NaN       NaN  False      -0.